# Advanced 08 — Proactive Agents

> A raw event is not an action. This lab turns signals into bounded proposals through application-owned admission, durable state, policy, authorization, routing, and delivery.

**Scenario:** Northstar Commerce monitors EU checkout. The credential-free fixture sends no real notification and performs no production mutation.

## Learning objectives and architecture

By the end you can distinguish event identity from correlation and delivery identity; keep hysteresis, debounce, cooldown, rate limiting, and backpressure separate; route with current on-call and IANA time; reconcile unknown delivery; and evaluate usefulness without hiding critical events.

```text
event/deadline/trend/prediction
→ admission → correlation/history → trigger/severity → atomic action dedupe
→ typed proposal → authorization/routing → durable delivery
→ confirmed | unknown/reconcile | failed/fallback/DLQ
```

## Part 1 — What proactive actually means

Proactivity begins from an environmental signal, schedule, absence, trend, or prediction rather than a direct user prompt. It does not enlarge authority. Event, trigger decision, notification permission, remediation permission, and execution are separate artifacts and controls.

## Reproducible setup

`policy.py` owns strict Pydantic v2 contracts and pure decisions. `lab.py` owns the durable SQLite fixture. Tests and notebook import exactly these modules. Trigger policies declare capability requirements; the engine receives independent, application-owned actor grants. A requirement is never treated as a grant.

In [ ]:
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

COURSE_DIR = Path("curriculum/advanced/08-proactive-agents").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

from policy import (
    BackpressureAction, BackpressurePolicy, DeliveryStatus, EventStatus,
    EventType, ProactiveActionType, ProactivePolicyError, RoutingAction,
    SensorQuality, Severity, TriggerStatus,
)
from lab import (
    FIXED_TIME, TENANT, DurableProactiveStore, ProactiveEngine,
    default_preference, default_trigger_policy, deliver_notification,
    demo_summary, evaluation_fixture, metric_event, on_call_assignment,
)

print("Credential-free deterministic mode: ON")
print("Production side effects: 0")

## Part 2 — Trusted event admission

The envelope binds event ID, source and source version, tenant, typed event, event/receipt times, payload digest, correlation key, sequence, safe facts, and schema version. `admit_event()` checks authenticated tenant, allow-listed source version, digest, future time, and maximum lateness before incident mutation.

In [ ]:
store = DurableProactiveStore()
engine = ProactiveEngine(store)
foreign = metric_event("foreign-1", value=0.7, event_time=FIXED_TIME, sequence=1, tenant_id="globex-commerce")
rejected = engine.process_event(
    foreign, policy=default_trigger_policy(), preference=default_preference(),
    on_call=on_call_assignment(), now=FIXED_TIME,
)
assert rejected.event_status is EventStatus.REJECTED
assert rejected.terminal_reason == "EVENT_TENANT_DENIED"
rejected.model_dump(mode="json")

## Part 3 — Event identity and atomic dedupe

`event_id` identifies one producer event. A versioned fingerprint identifies a stable semantic noise class and makes a time-bounded atomic claim; it excludes exact timestamp and raw numeric severity inputs. Derived severity remains separate so a material P3→P1 change can explicitly bypass suppression. A logical notification ID remains stable across delivery retries; attempt IDs remain unique. Dedupe reduces duplicates but cannot promise exactly once.

## Part 4 — Incident correlation

A correlation key groups distinct evidence into a versioned incident. The correlation window expires independently of dedupe. Incident history is established before the fingerprint claim can return a duplicate, so a competing claim cannot strand an admitted event. An exact `event_id` transport redelivery increments duplicate-delivery count but not operational occurrence count; a distinct source event may increment occurrence count even when its notification is deduplicated. Out-of-order events cannot regress state, and delayed evidence cannot reopen a resolved incident.

In [ ]:
store = DurableProactiveStore()
engine = ProactiveEngine(store)
event = metric_event("source-event-1", value=0.40, event_time=FIXED_TIME, sequence=1)
kwargs = dict(policy=default_trigger_policy(), preference=default_preference(timezone="UTC"), on_call=on_call_assignment(), now=FIXED_TIME)
first = engine.process_event(event, **kwargs)
duplicate = engine.process_event(event, **kwargs)
incident = store.get_incident(TENANT, first.incident_id)
assert duplicate.event_status is EventStatus.DUPLICATE
assert (incident.occurrence_count, incident.duplicate_delivery_count) == (1, 1)
{"first": first.terminal_reason, "duplicate": duplicate.terminal_reason, "incident": incident.model_dump(mode="json")}

## Part 5 — Hysteresis, debounce, and cooldown

Hysteresis uses separate activation/recovery thresholds. Debounce requires continuous event-time duration. Cooldown limits repeat notification. Missing or invalid sensor data breaks a streak. Sustained healthy readings resolve an active incident.

## Part 6 — Trigger and severity policy

Severity is derived from typed facts, never free text. A new P1 bypasses older suppression. Trigger policy chooses among notify, create task, request approval, read-only investigation, pre-authorized workflow, defer, and suppress.

In [ ]:
store = DurableProactiveStore()
engine = ProactiveEngine(store)
states = []
for sequence, seconds in enumerate((0, 60, 120), start=1):
    at = FIXED_TIME + timedelta(seconds=seconds)
    states.append(engine.process_event(
        metric_event(f"breach-{sequence}", value=0.40, event_time=at, sequence=sequence),
        policy=default_trigger_policy(), preference=default_preference(timezone="UTC"),
        on_call=on_call_assignment(valid_until=FIXED_TIME + timedelta(hours=2)), now=at,
    ))
assert [s.trigger_status for s in states] == [TriggerStatus.PENDING, TriggerStatus.PENDING, TriggerStatus.ACTIVATED]
[s.terminal_reason for s in states]

## Part 7 — Typed notification proposal

A proposal binds tenant, incident, recipient scope, severity, category, evidence, deadline, capability, and policy version. It grants no remediation authority. `TriggerPolicy.required_capabilities` states what is required; `ProactiveEngine.actor_capabilities` states what the execution principal actually has. Editing the policy cannot widen authority.

## Part 8 — Quiet hours, timezone, and preferences

Routing uses IANA timezones and current on-call state. P4 work becomes a durable next-workday digest outside local hours. Explicit organizational P1 policy overrides optional quiet hours and opt-out.

In [ ]:
proposal, routing, row = store.load_notification(TENANT, states[-1].notification_id)
assert proposal.required_capability == "notify.oncall"
assert routing.recipient_id == "northstar-primary-1"
assert row["status"] in {"ROUTED", "DEFERRED"}
{"proposal": proposal.model_dump(mode="json"), "routing": routing.model_dump(mode="json")}

## Part 9 — Acknowledgment and escalation

Acknowledgment is a state transition that cancels pending escalation. An unacknowledged P1 advances through durable roles by creating a typed proposal and using the same routing, provider-attempt, receipt, fallback, and audit controls.

## Part 10 — Digest lifecycle

Deferred items survive restart. Resolution changes the morning summary to ‘occurred and resolved’; superseded or cancelled work is not sent as active. `dispatch_digest()` is an aggregation-only fixture, not provider delivery.

## Part 11 — Idempotent delivery

Retries preserve logical identity and use one unique, persisted attempt ID per external provider call. Confirmed delivery is a no-op on replay. An unknown provider result must be reconciled before retry.

## Part 12 — Retries, fallback, and dead letters

Transient P1 failures can use an authorized fallback channel. The failed primary and fallback are separately accounted attempts. Retries are bounded; the fixture does not model backoff/jitter, and exhausted work is dead-lettered rather than retried forever.

In [ ]:
receipt = deliver_notification(
    store, tenant_id=TENANT, logical_notification_id=states[-1].notification_id,
    preference=default_preference(timezone="UTC"), current_on_call=on_call_assignment(valid_until=FIXED_TIME + timedelta(hours=2)),
    provider_outcomes={"slack": DeliveryStatus.DELIVERED}, now=FIXED_TIME + timedelta(seconds=130),
)
replayed = deliver_notification(
    store, tenant_id=TENANT, logical_notification_id=states[-1].notification_id,
    preference=default_preference(timezone="UTC"), current_on_call=on_call_assignment(valid_until=FIXED_TIME + timedelta(hours=2)),
    provider_outcomes={}, now=FIXED_TIME + timedelta(seconds=131),
)
assert replayed == receipt
receipt.model_dump(mode="json")

## Part 13 — Rate limiting and backpressure

Recipient/channel budgets constrain low-priority floods with a typed, audited `DEFERRED` result. Queue depth and consumer lag decide whether lower-priority work is processed, aggregated, or shed. Shed events record tenant, severity, reason, count, and policy version; shed-event rate makes recall impact visible. P1 remains preserved, and correlated events should be summarized before optional model work.

## Part 14 — Read-only proactive investigation

Missing expected backups and trends can propose a bounded read-only investigation. `notify.oncall` never implies workflow or production-write capability.

In [ ]:
bp = BackpressurePolicy(tenant_id=TENANT, batch_depth=10, shed_depth=20, maximum_consumer_lag_seconds=60)
low = metric_event("low", value=0.05, affected_customer_pct=0, event_time=FIXED_TIME, sequence=1)
critical = metric_event("critical", value=0.70, affected_customer_pct=70, event_time=FIXED_TIME, sequence=2)
pressure_store = DurableProactiveStore()
pressure_engine = ProactiveEngine(pressure_store)
low_decision = pressure_engine.apply_backpressure(low, queue_depth=30, consumer_lag_seconds=90, policy=bp, now=FIXED_TIME)
critical_decision = pressure_engine.apply_backpressure(critical, queue_depth=30, consumer_lag_seconds=90, policy=bp, now=FIXED_TIME)
assert low_decision.action is BackpressureAction.SHED
assert critical_decision.action is BackpressureAction.PROCESS_NOW
assert sum(row["event_type"] == "EVENT_SHED" for row in pressure_store.audit_events(TENANT)) == 1
{"low": low_decision.model_dump(mode="json"), "critical": critical_decision.model_dump(mode="json")}

## Part 15 — Prompt injection and tenant isolation

Free text cannot select severity, recipient, capability, policy, or terminal state. Cross-tenant event, preference, incident, digest, and delivery access fails closed. Audit records keep safe metadata digests instead of arbitrary payload text.

## Part 16 — Restart and durability

SQLite persists processed events, atomic claims, incident versions, samples, notifications, provider attempts, digests, escalations, rate counters, per-incident model usage, dead letters, and audit. The model-call budget therefore survives engine restart. `state_version` plus `expected_version` rejects stale concurrent writes.

## Part 17 — Evaluation against a same-task baseline

The labelled fixture compares naive `event → model → notification` with the governed pipeline. It measures trigger precision/recall, P1 miss rate, notification precision, duplicates, detection/delivery latency, model calls, and estimated cost. These deterministic outputs validate mechanics, not production generalization.

In [ ]:
metrics = evaluation_fixture()
assert metrics["governed"].notification_precision > metrics["naive"].notification_precision
assert metrics["governed"].model_calls < metrics["naive"].model_calls
assert metrics["governed"].p1_miss_rate == 0
{name: value.model_dump() | {"trigger_precision": value.trigger_precision, "trigger_recall": value.trigger_recall, "notification_precision": value.notification_precision} for name, value in metrics.items()}

## Part 18 — Optional platform adapters

Kafka and EventBridge are widely used transport/routing options. Redis is one widely used low-latency state option. They are adapters: application code still owns admission, correlation, severity, capabilities, routing, delivery state, and completion. The core lab needs no external credentials.

In [ ]:
summary = demo_summary()
assert summary["production_side_effects"] == 0
assert summary["model_calls_on_critical_path"] == 0
summary

## Exercises

1. Add scope expansion as a material change that bypasses cooldown, then write a regression test.
2. Add a certificate-expiry policy that creates a renewal task but cannot execute a production workflow.
3. Change the Vancouver quiet-hours fixture to a fall-back DST date and predict the UTC delivery time before running it.
4. Add provider reconciliation that returns ‘not found’ and permits exactly one new attempt.
5. Design an evaluation split that detects whether a more aggressive shedding policy harms P1 recall.

### Final checkpoint

Explain why dedupe is not exactly once, why severity escalation bypasses old suppression, why P1 routing cannot depend on a model, why on-call must be resolved at send time, and which metrics demonstrate usefulness rather than mere quietness.